# GROTH-07 : la capacité — le cœur de la preuve formelle du trou de Smoktunowicz

**Navigation** : [Index de la série](../README.md) · [Îlot Grothendieck](README.md) · [< GROTH-06 Groupes formels](GROTH-06-FormalGroups-Lean.ipynb) | [Compagnon Lean](GROTH-07-Capacity-Lean.ipynb)

**Kernel** : Python 3 (les énoncés Lean sont compilés par le
[compagnon Lean](GROTH-07-Capacity-Lean.ipynb), sous kernel Lean 4)

***

Ce carnet ne s'attarde pas sur le **comptage** de la fonction de Hilbert dans les
familles classiques : il s'occupe de ce que la **preuve formelle** du trou en fait.
Son objet est la **capacité** — la quantité que la preuve formelle introduit, et
qui n'apparaît dans aucune des publications de référence. Le cadre (dimension de
Gelfand-Kirillov, énoncé du trou) est repris ici juste ce qu'il faut pour rendre
cette quantité lisible, et le compagnon Lean en donne l'ancrage formel.

## Objectifs d'apprentissage

À la fin de ce carnet, vous saurez :

1. **Définir** la dimension de Gelfand-Kirillov et la **calculer** sur un anneau
   concret, puis **modéliser** le mécanisme de la preuve formelle du trou ;
2. **Énoncer** le théorème du trou de Smoktunowicz et **situer** les exclusions qui le précèdent ;
3. **Expliquer** l'idée de la *capacité* et la **mettre en œuvre** sur un modèle fini ;
4. **Nommer** ce que la bibliothèque de maths de Lean sait faire, et ce qu'elle ne sait pas encore faire.

## Prérequis

- Algèbre linéaire sur le corps à deux éléments : vecteur, base, dimension, sous-espace.
- Anneau commutatif : produit, sous-espaces vectoriels d'un anneau.
- Aucune connaissance préalable en algèbre non commutative n'est requise.

## Durée estimée

45 minutes de lecture active, dont une vingtaine de manipulation du code.

## 1. Le cadre : la dimension de Gelfand-Kirillov

Soit `A` une algèbre sur un corps, engendrée par un nombre fini d'éléments. On note
`A_n` l'espace des éléments obtenus par **au plus `n` multiplications** des générateurs.
La suite `dim A_n` est croissante : c'est la *fonction de Hilbert* de l'algèbre.

Si cette suite croît comme `n^d`, on dit que `A` est de **croissance polynomiale de
degré `d`**, et `d` est sa **dimension de Gelfand-Kirillov**, notée `GKdim A`.

Deux repères :

- une algèbre commutative comme `K[X_1, …, X_d]` a `GKdim = d` ;
- en général, la dimension de Gelfand-Kirillov peut prendre **n'importe quelle valeur
  réelle à partir de 2** : c'est une quantité non entière, et c'est précisément ce qui
  rend le théorème de la section 2 non trivial.

Le calcul se fait sur la fonction de Hilbert : dans `d` variables, les monômes de degré
total au plus `n` sont les multi-exposants dont la somme ne dépasse pas `n`, il y en a
donc `C(n+d, d)`.


In [1]:
from math import comb, log

# Dimension de l'espace des polynômes de degré total <= n en d variables.
# Les monômes sont les multi-exposants (a_1, ..., a_d) avec a_1 + ... + a_d <= n ;
# par le principe des tiroirs (billes dans des boîtes), il y en a C(n+d, d).
def hilbert_poly(d: int, n: int) -> int:
    return comb(n + d, d)


# Affichage de la fonction de Hilbert pour quelques nombres de variables.
print("fonction de Hilbert de K[X_1, ..., X_d] :")
for d in (1, 2, 3):
    valeurs = [hilbert_poly(d, n) for n in range(9)]
    print(f"  d = {d} : H(n) = {valeurs}")


fonction de Hilbert de K[X_1, ..., X_d] :
  d = 1 : H(n) = [1, 2, 3, 4, 5, 6, 7, 8, 9]
  d = 2 : H(n) = [1, 3, 6, 10, 15, 21, 28, 36, 45]
  d = 3 : H(n) = [1, 4, 10, 20, 35, 56, 84, 120, 165]


### Lecture du résultat : la fonction de Hilbert

Les valeurs obtenues sont exactement les coefficients binomiaux `C(n+d, d)` : la
fonction de Hilbert d'un anneau polynomial est un polynôme en `n` de degré `d`. C'est
déjà le signal que la croissance est polynomiale — reste à en mesurer l'exposant.

In [2]:
def exposant_croissance(G, n_min: int) -> float:
    # Exposant mesuré par la pente log-log : (log G(n) - log G(n-1)) / log(n/(n-1)).
    # Pour une suite G(n) ~ c * n^d, cette pente tend vers d.
    n = n_min
    return (log(G(n)) - log(G(n - 1))) / log(n / (n - 1))


# Mesure de l'exposant pour d = 1, 2, 3 : la pente se stabilise sur la valeur d.
print("exposant mesuré (d attendu = nombre de variables) :")
for d in (1, 2, 3):
    G = lambda n, d=d: hilbert_poly(d, n)
    mesures = [round(exposant_croissance(G, n), 4) for n in (16, 32, 64, 128)]
    print(f"  d = {d} : {mesures}")


exposant mesuré (d attendu = nombre de variables) :
  d = 1 : [0.9394, 0.9692, 0.9845, 0.9922]
  d = 2 : [1.825, 1.9095, 1.954, 1.9768]
  d = 3 : [2.6628, 2.8225, 2.9088, 2.9538]


### Lecture du résultat : l'exposant converge vers le nombre de variables

Les trois séries de mesures tendent vers la valeur attendue, de plus en plus vite à
mesure que l'on observe la croissance plus loin. C'est la mesure directe de la
dimension de Gelfand-Kirillov, et c'est le sens de la définition : **l'exposant de la
croissance est la dimension**.

Le compagnon Lean de ce carnet établit le cas `d = 1` par une autre voie — non par
une mesure numérique, mais par les deux bornes qui caractérisent un exposant.

## 2. Le théorème du trou de Smoktunowicz

**Énoncé (Smoktunowicz, 2006).** Soient `A` une algèbre de type fini sur un corps,
graduée par `N`, sans diviseurs de zéro, et dont la partie de degré `0` est de
dimension finie. Alors la dimension de Gelfand-Kirillov de `A` est **soit au plus 2,
soit au moins 3**. La valeur 3 est exclue autant que 2, et aucune valeur strictement
comprise entre 2 et 3 n'est possible.

Autrement dit : pour cette classe d'algèbres, les taux de croissance possibles
sautent d'un côté à l'autre de l'intervalle `]2, 3[`. C'est un **trou** dans l'ensemble
des dimensions réalisables.

L'exclusion est progressive, et c'est cette construction qui rend le théorème digne
d'intérêt :

| Résultat | Hypothèses supplémentaires | Exclut |
|---|---|---|
| Bergman (2000, Thm 2.5) | aucune (algèbre de type fini) | `]1, 2[` |
| Artin-Stafford (1995) | graduée par `N`, sans diviseurs de zéro | `]2, 11/5[` |
| Smoktunowicz (2006) | + partie de degré `0` de dimension finie | `]2, 3[` |

Le théorème de Smoktunowicz **absorbe** celui d'Artin-Stafford et renforce celui de
Bergman : il ne s'agit pas d'une troisième exclusion indépendante, mais d'une extension
à l'intervalle `]2, 3[` complet.


In [3]:
# Exclusions successives, dans l'ordre de leur obtention.
INTERDITS = [
    ("Bergman (2000, Thm 2.5)", 1.0, 2.0),
    ("Artin-Stafford (1995)", 2.0, 11 / 5),
    ("Smoktunowicz (2006, Thm 1)", 2.0, 3.0),
]

print("dimensions de Gelfand-Kirillov exclues par résultat écrit :")
for nom, a, b in INTERDITS:
    print(f"  {nom:28s} : ] {a:.4g} , {b:.4g} [  exclu")

# Conséquence combinée pour la classe des algèbres de type fini graduées par N,
# sans diviseurs de zéro, de partie de degré 0 de dimension finie.
print()
print("  valeur   statut pour la classe du théorème de Smoktunowicz")
for x in (0.5, 1.0, 1.5, 2.0, 2.2, 2.5, 2.9, 3.0, 3.5, 4.0):
    exclu = any(a < x < b for _, a, b in INTERDITS)
    print(f"  {x:5.1f}    {'impossible' if exclu else 'possible'}")


dimensions de Gelfand-Kirillov exclues par résultat écrit :
  Bergman (2000, Thm 2.5)      : ] 1 , 2 [  exclu
  Artin-Stafford (1995)        : ] 2 , 2.2 [  exclu
  Smoktunowicz (2006, Thm 1)   : ] 2 , 3 [  exclu

  valeur   statut pour la classe du théorème de Smoktunowicz
    0.5    possible
    1.0    possible
    1.5    impossible
    2.0    possible
    2.2    impossible
    2.5    impossible
    2.9    impossible
    3.0    possible
    3.5    possible
    4.0    possible


### Lecture du résultat : ce que le théorème exclut

Le tableau final se lit comme une **carte des valeurs interdites** : tout ce qui est
strictement entre 1 et 2 est exclu par le résultat le plus ancien, et tout ce qui est
strictement entre 2 et 3 l'est par le plus récent. Les bornes 2 et 3, elles, ne sont
pas exclues — elles restent les deux seuls comportements possibles au voisinage du trou.

Le trou n'est pas une curiosité : il restreint la classification des domaines gradués
à croissance polynomiale, et il rend le passage à la dimension **exacte** (`GKdim = 2`
plutôt que `GKdim < 2`) un problème délicat — c'est là que les preuves publiées
s'appuient sur des résultats profonds.

## 3. L'idée qui change tout : la capacité

Les preuves publiées mobilisent des résultats profonds d'algèbre et de géométrie
algébrique, absents de la bibliothèque de maths de Lean. La preuve formelle évite ces
résultats et introduit une quantité **nouvelle** : la **capacité**.

Le dispositif, en langage géométrique. On se place dans une pièce `A_N` d'un anneau
gradué `A` sur le corps à deux éléments, et on fixe la pièce finie `S = span(1, X)`
par laquelle on fait croître les sous-espaces — le rôle que tient dans la preuve la
pièce `S` qui engendre le corps de division :

- pour un sous-espace `U ⊆ A_N`, on mesure son **défaut de croissance** sous la
  multiplication par `S` : `dim(S·U) - dim(U)` — le produit peut déborder d'un degré
  (`S·U ⊆ A_{N+1}`), et le défaut mesure de combien l'espace grandit ;
- la **capacité** de `U` est le plus petit de ces défauts parmi les sous-espaces `W`
  contenant `U` — donc le coût minimal qu'il faut payer pour faire croître `U` ;
- le point décisif est que les éléments qui **n'augmentent pas** la capacité forment
  un **corps de division** : il suffit donc de contrôler un ensemble de générateurs
  pour contrôler tout l'anneau.

L'image juste est celle d'une **frontière** : la capacité d'un sous-espace se comporte
comme la taille d'une frontière dans un graphe, et les éléments qui ne l'augmentent
pas sont ceux qui restent « à l'intérieur ».

Le point de méthode est riche : la quantité de base est la **différence de volume** qui
définit le degré de transcendance inférieure de Zhang (1998), et l'argument de
stabilisateur qui fait apparaître un corps de division y a déjà un équivalent. Ce qui
est propre à la preuve formelle est l'usage systématique de la minimisation sur les
sous-espaces plus grands, et l'argument de « plateau » bâti dessus.

Attention au statut : il s'agit d'une **preuve formelle du théorème**, dont l'argument
central n'apparaît dans aucune des trois publications de référence. La preuve publiée
et la preuve formelle ne se rejoignent que sur les premières étapes.

Ce que nous allons faire ici est plus modeste, et c'est ce qui rend la section
manipulable : **vérifier le mécanisme sur un modèle fini**.


In [4]:
# Modèle fini de l'idée de capacité.
# A = F_2[X] gradué par le degré ; A_N = polynômes de degré <= N, de dimension N+1.
# S = span(1, X) est la pièce finie par laquelle on fait croître les sous-espaces.
N = 3
DIM_A = N + 1          # dimension de la pièce A_N
MOD = 1 << DIM_A       # vecteurs de F_2 de dimension DIM_A (bits 0..N)


def mult(a: int, b: int) -> int:
    # Produit dans F_2[X], tronqué au degré N+1 : S*U doit pouvoir déborder
    # d'un degré, car S*U est contenu dans A_{N+1} et non dans A_N.
    r = 0
    for i in range(DIM_A):
        if (a >> i) & 1:
            for j in range(DIM_A):
                if (b >> j) & 1 and i + j <= N + 1:
                    r ^= 1 << (i + j)
    return r


def dim_F2(elems) -> int:
    # Rang sur F_2 d'un ensemble de vecteurs (chacun un masque de bits).
    # Chaque pivot porte un bit directeur distinct ; la réduction par XOR
    # annule ce bit, d'où le min() numérique.
    pivots = []
    for v in elems:
        cur = v
        for p in pivots:
            cur = min(cur, cur ^ p)
        if cur:
            pivots.append(cur)
            pivots.sort(reverse=True)
    return len(pivots)


def sous_espaces():
    # Énumération de tous les sous-espaces de A_N : un sous-espace engendré par
    # {v_1, ..., v_k} est atteint par k ajouts successifs d'un vecteur, et tout
    # sous-espace s'obtient ainsi.
    elements = list(range(MOD))
    connus = {frozenset({0})}
    change = True
    while change:
        change = False
        for T in list(connus):
            for v in elements:
                if v not in T:
                    U = T | {x ^ v for x in T}
                    if U not in connus:
                        connus.add(U)
                        change = True
    return sorted(connus, key=lambda T: (dim_F2(T), sorted(T)))


def produit(S, W):
    # Sous-espace engendré par les produits s * w, avec s dans S et w dans W.
    return frozenset(mult(s, w) for s in S for w in W)


espaces = sous_espaces()
print(f"pièce A_N sur F_2[X] (N = {N}) : dimension {DIM_A}, "
      f"{len(espaces)} sous-espaces")


pièce A_N sur F_2[X] (N = 3) : dimension 4, 67 sous-espaces


### Lecture du résultat : les sous-espaces de la pièce graduée

L'énumération par ajouts successifs est exhaustive : tout sous-espace est engendré par
une famille finie de vecteurs, donc atteint en un nombre fini d'ajouts. La pièce `A_N`
est de dimension `N+1`, ce qui en fait un objet fini dont on peut **parcourir**
intégralement les sous-espaces — condition nécessaire pour qu'une minimisation soit
effectivement calculable.

In [5]:
S = frozenset({0, 1, 1 << 1})   # span de 1 et de X


def defaut(W):
    # Défaut de croissance de W : dim(S*W) - dim(W).
    return dim_F2(produit(S, W)) - dim_F2(W)


def capacite(U):
    # Capacité de U : le plus petit défaut parmi les sous-espaces W contenant U.
    return min(defaut(W) for W in espaces if U <= W)


capacites = {U: capacite(U) for U in espaces}

# Distribution des capacités sur l'ensemble des sous-espaces.
compteur = {}
for c in capacites.values():
    compteur[c] = compteur.get(c, 0) + 1
print("distribution des capacités (valeur -> nombre de sous-espaces) :")
for c in sorted(compteur):
    print(f"  capacité {c} : {compteur[c]} sous-espaces")

# Monotonie : si U est contenu dans W, alors cap(U) <= cap(W), car l'ensemble
# des candidats de cap(U) est plus grand que celui de cap(W).
violations = sum(1 for U in espaces for W in espaces
                 if U <= W and capacites[U] > capacites[W])
print()
print("monotonie vérifiée sur toutes les paires de sous-espaces :", violations == 0)


distribution des capacités (valeur -> nombre de sous-espaces) :
  capacité 0 : 1 sous-espaces
  capacité 1 : 66 sous-espaces

monotonie vérifiée sur toutes les paires de sous-espaces : True


### Lecture du résultat : une capacité calculable, mais grossière sur une pièce si petite

La distribution mesurée ne prend que deux valeurs : la capacité vaut `0` pour l'unique
sous-espace nul, et `1` pour les `66` autres sous-espaces de la pièce — soit `67`
sous-espaces au total.

Il faut donc être précis sur ce que le modèle **ne** montre pas : sur une pièce de
dimension `4`, la capacité n'est pas plus fine que la dimension — le seul sous-espace
de capacité `0` est le sous-espace nul, et tous les autres se valent. Le modèle ne
sépare donc pas deux sous-espaces de même dimension.

Ce qu'il établit en revanche est vérifiable ici par examen exhaustif : la capacité est
**calculable** — une minimisation sur les sous-espaces plus grands, faite sur la pièce
entière — et elle est **monotone**, propriété attendue d'une quantité de type
« frontière » : plus on broute le sous-espace, moins il reste de marge pour le faire
croître. La monotonie est ici vérifiée sur **toutes** les paires, pas sur un
échantillon.

Ce que le modèle ne fait pas — et c'est le cœur du théorème — c'est établir que les
éléments préservant la capacité forment un corps de division. C'est cette propriété
qui permet de ramener le contrôle d'un anneau à celui de quelques générateurs, et
c'est elle qui manque à ce modèle fini, où la structure de corps de division n'est pas
disponible. Le passage de « calculable et monotone » à « discriminante » demande une
pièce plus grande : c'est la limite assumée de la démonstration par modèle fini.

## 4. La frontière de ce que sait faire Lean

La bibliothèque de maths de Lean ne sait pas encore ce dont ce carnet parle. La
frontière se situe à plusieurs niveaux, et il est utile de les nommer explicitement
plutôt que de parler d'un « manque » indistinct.

| Brique | État dans la bibliothèque de Lean |
|---|---|
| Monïdes gradués | **présent** (`GradedMonoid`) |
| Dimension de Gelfand-Kirillov | **absente** — ni fonction ni théorème |
| Anneaux gradués au sens algébrique | **absent** — pas d'algèbre graduée indexée par `N` |
| Localisation d'Ore | **absente** — pas de corps de fractions d'un anneau non commutatif |
| Théorème de Bergman (exclusion `]1, 2[`) | **absent** |
| Argument de Riemann-Roch de la preuve publiée | **absent** |

Autrement dit, la preuve formelle ne s'appuie sur aucun de ces résultats : elle
contourne la frontière au lieu de la franchir. C'est ce qui la rend remarquable, et
c'est aussi ce qui la rend fragile : une telle preuve est un passage forcé, dont la
solidité tient à la relecture indépendante du noyau.

Le compagnon Lean de ce carnet ([`GROTH-07-Capacity-Lean.ipynb`](GROTH-07-Capacity-Lean.ipynb),
lake [`gkdim_lean/`](gkdim_lean/)) se place **en deçà** de la frontière : il ne
formalise pas le théorème du trou, mais il vérifie le cas de référence — la croissance
exactement linéaire de l'anneau polynomial à une variable, donc une dimension de
Gelfand-Kirillov égale à 1 — par les deux bornes qui caractérisent un exposant. Il
fournit le premier jalon, pas la frontière.


In [6]:
# Point sur le compagnon Lean : les déclarations qu'il établit.
from pathlib import Path


def trouver(nom: str):
    # Cherche un fichier dans le répertoire courant, puis dans ses parents.
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidat = base / nom
        if candidat.exists():
            return candidat
    return None


lacine = trouver("gkdim_lean/GKdim/Core.lean")
if lacine is None:
    print("compagnon Lean non trouvé depuis le dossier du carnet")
    print("attendu : gkdim_lean/GKdim/Core.lean (relatif au dossier du carnet)")
else:
    source = lacine.read_text(encoding="utf-8")
    declarations = [l for l in source.splitlines() if l.startswith(("def ", "theorem "))]
    ancetre = next((p for p in lacine.parents if p.name == "Grothendieck"), None)
    relatif = lacine.relative_to(ancetre).as_posix() if ancetre else lacine.name
    print("compagnon Lean :", relatif)
    for d in declarations:
        print("   " + d)
    print()
    print("fichiers du module :", sorted(p.name for p in lacine.parent.glob("*.lean")))


compagnon Lean : gkdim_lean/GKdim/Core.lean
   def hilbert_poly1 (n : ℕ) : ℕ := n + 1
   def growthAtMost (G : ℕ → ℕ) (d : ℕ) : Prop :=
   def growthAtLeast (G : ℕ → ℕ) (d : ℕ) : Prop :=
   theorem poly1_growthAtMost : growthAtMost hilbert_poly1 1 := by
   theorem poly1_growthAtLeast : growthAtLeast hilbert_poly1 1 := by
   theorem poly1_gkdim_is_one :

fichiers du module : ['Core.lean', 'Core_en.lean']


### Lecture du résultat : ce que le compagnon Lean établit

Le module companion expose exactement deux choses : les **deux bornes** de croissance
(`growthAtMost` et `growthAtLeast`), et le **jeu de lemmes** qui montre que la
fonction de Hilbert de l'anneau polynomial à une variable est bornée au-dessus par un
multiple de `n` et en-dessous par `n` lui-même.

Le théorème final du module — la croissance est exactement linéaire, donc la dimension
de Gelfand-Kirillov vaut 1 — est la **forme jouable** de la définition de la section 1 :
les deux bornes sont l'ancrage formel de l'exposant mesuré numériquement plus haut.

Les deux fichiers du module sont les jumeaux FR et EN de la convention i18n du dépôt :
le code est identique, seules les docstrings diffèrent.

## 5. Exercices

Les trois exercices ci-dessous sont des **stubs à compléter**. Aucun ne reprend
l'énoncé d'un exemple guidé : le premier mesure une grandeur (l'exposant de
croissance), le second formule une propriété (les deux bornes), le troisième
reconstruit un minimum (la capacité). Ils sont volontairement ordonnés du plus direct
au plus exigeant.

### Exercice 1 : mesurer la dimension de Gelfand-Kirillov

**Objectif** : écrire une fonction `dimension_gk(d)` qui renvoie l'exposant de
croissance mesuré de l'anneau `K[X_1, …, X_d]`, et le comparer à la valeur attendue
`d`. On réutilisera `hilbert_poly` et la pente log-log.

**Indication** : la pente mesurée converge vite ; une seule valeur de `n` suffit
pour distinguer `d = 1` de `d = 3`.


In [7]:
# Exercice 1 : mesurer la dimension de Gelfand-Kirillov de K[X_1, ..., X_d]
# TODO etudiant : completer dimension_gk pour qu'elle renvoie l'exposant de
#   croissance mesure de l'anneau polynomial a d variables (valeur attendue : d)
# Indice : reutiliser hilbert_poly et la pente log-log ; verifier sur d = 1, 2, 3
# Etape 1 : choisir un seuil n et calculer la pente entre n et n-1
# Etape 2 : renvoyer l'exposant mesure

resultat = None  # TODO etudiant
print("dimension_gk a completer :", resultat)


dimension_gk a completer : None


### Exercice 2 : formuler les deux bornes de croissance

**Objectif** : écrire une fonction `croissance_lineaire(G)` qui exprime qu'une suite
`G : N -> N` a une croissance **exactement linéaire** : bornée au-dessus par un
multiple de `n`, et bornée en-dessous par `c * n` avec `c > 0`. C'est l'énoncé que le
compagnon Lean établit pour l'anneau polynomial à une variable.

**Indication** : la borne supérieure admet la constante `C = 2` dès `n >= 1` ; la borne
inférieure admet `c = 1`. Vérifier sur `G = lambda n: n + 1`, puis sur une suite qui
croît comme `n^2` — celle-ci doit échouer à la borne inférieure de degré 1.


In [8]:
# Exercice 2 : formuler les deux bornes de croissance
# TODO etudiant : ecrire croissance_lineaire(G) au sens du compagnon Lean
#   (une borne superieure de degre 1 et une borne inferieure de degre 1)
# Indice : borne superieure C = 2 pour n >= 1 ; borne inferieure c = 1
# Etape 1 : tester la borne superieure
# Etape 2 : tester la borne inferieure

resultat = None  # TODO etudiant
print("croissance_lineaire a completer :", resultat)


croissance_lineaire a completer : None


### Exercice 3 : la capacité comme coût de croissance

**Objectif** : compléter `capacite(U)`, qui renvoie le plus petit défaut
`dim(S·W) - dim(W)` sur les sous-espaces `W` contenant `U`, avec `S = span(1, X)`. On
réutilisera les sous-espaces déjà énumérés et la fonction `dim_F2`.

**Indication** : filtrer d'abord les `W` contenant `U`, puis calculer le produit `S·W`
et son défaut ; l'anneau est commutatif, la multiplication par `S` est donc commutative
et l'ordre des facteurs n'a pas d'importance.


In [9]:
# Exercice 3 : la capacite comme cout de croissance
# TODO etudiant : completer capacite(U) (minimum de dim(S*W) - dim(W) sur W >= U)
# Indice : reutiliser les sous-espaces deja enumeres et dim_F2 ; le produit
#   non commutatif n'est pas necessaire ici, l'algebre est commutative
# Etape 1 : filtrer les W contenant U
# Etape 2 : calculer le produit S*W et son defaut

resultat = None  # TODO etudiant
print("capacite a completer :", resultat)


capacite a completer : None


## 6. Conclusion

Ce carnet a défini la dimension de Gelfand-Kirillov comme l'exposant de croissance de
la fonction de Hilbert, l'a **mesurée** sur un anneau polynomial, a situé le
théorème du trou de Smoktunowicz parmi les exclusions qui le précèdent, et a mis en
œuvre sur un modèle fini le mécanisme de la **capacité** — un défaut de croissance
minimisé sur les sous-espaces plus grands.

### Récapitulatif

| Notion | Contenu | Où c'est établi |
|---|---|---|
| Dimension de Gelfand-Kirillov | exposant de la croissance de `dim A_n` | section 1 (mesure) et compagnon Lean (bornes formelles) |
| Trou de Smoktunowicz | `GKdim ≤ 2` ou `GKdim ≥ 3` pour les algèbres graduées de type fini sans diviseurs de zéro | section 2 (tableau des exclusions) |
| Capacité | plus petit défaut `dim(S·W) - dim(W)` sur les `W` contenant `U` | section 3 (modèle fini) |
| Frontière Lean | `GradedMonoid` présent, `GKdim` absent | section 4 |

### Pour aller plus loin

- Le lac [`gkdim_lean/`](gkdim_lean/) porte le compagnon formel ; son
  [README](gkdim_lean/README.md) décrit les conventions suivies.
- Le parcours complet de l'îlot : [GROTH-01](GROTH-01-Categories-Mathlib-Lean.ipynb)
  (la marche manquante) → [GROTH-04](GROTH-04-Companion-Lean.ipynb) (companion
  formel) → [GROTH-06](GROTH-06-FormalGroups-Lean.ipynb) (groupes formels).

**Navigation** : [<< GROTH-06 Groupes formels](GROTH-06-FormalGroups-Lean.ipynb) |
[Compagnon Lean](GROTH-07-Capacity-Lean.ipynb) |
[Îlot Grothendieck](README.md) | [Index de la série](../README.md)